# День 4 — Baseline без обучения трансформера

**Идея:** трансформер используется как «замороженный» экстрактор признаков — мы его не дообучаем. Учится только простая логистическая регрессия поверх CLS-эмбеддингов.

Это даёт **точку отсчёта**: если полноценный fine-tuning потом не обойдёт этот baseline, значит что-то пошло не так.

Переиспользуемый код — в [`baseline_utils.py`](baseline_utils.py). Датасет — **SST-2** (бинарная тональность рецензий на фильмы).

## Задача 1: Токенизация текстов

Это ровно та же функция, что и в Дне 1 — повторяем её здесь для полноты.

In [1]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_texts(texts, max_length=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )


sample_texts = [
    "This movie was absolutely amazing!",
    "Terrible movie, waste of time.",
    "Pretty good, I liked it.",
]

batch = tokenize_texts(sample_texts)
print(f"input_ids shape:      {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"\ninput_ids:\n{batch['input_ids']}")
print(f"\nattention_mask:\n{batch['attention_mask']}")

C:\Users\Vsevolod\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


input_ids shape:      torch.Size([3, 9])
attention_mask shape: torch.Size([3, 9])

input_ids:
tensor([[ 101, 2023, 3185, 2001, 7078, 6429,  999,  102,    0],
        [ 101, 6659, 3185, 1010, 5949, 1997, 2051, 1012,  102],
        [ 101, 3492, 2204, 1010, 1045, 4669, 2009, 1012,  102]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


## Задача 2: Извлечение CLS-эмбеддингов

In [2]:
from transformers import AutoModel
import torch
import numpy as np

model = AutoModel.from_pretrained(model_name)
model.eval()


def get_cls_embeddings(texts, batch_size=32):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        tokens = tokenize_texts(batch_texts)

        with torch.no_grad():
            outputs = model(**tokens)

        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)


test_emb = get_cls_embeddings(sample_texts)
print(f'Embeddings shape: {test_emb.shape}')
print(f'Ожидается: (3, 768)')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7593.42it/s]


[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings shape: (3, 768)
Ожидается: (3, 768)


## Задача 3: Logistic Regression на эмбеддингах

### 3.1 Загрузка датасета

Берём стратифицированную подвыборку в 3000 предложений — полный train SST-2 это 67k, и инференс на CPU занял бы слишком долго.

In [3]:
from baseline_utils import load_sst2

df = load_sst2(n_samples=3000, random_state=42)

print(f'Размер: {df.shape}')
print(f'\nБаланс классов (0=negative, 1=positive):')
print(df['label'].value_counts())
df.head()

Размер: (3000, 2)

Баланс классов (0=negative, 1=positive):
label
1    1673
0    1327
Name: count, dtype: int64


,text,label
0,a perfect family film,1
1,brings a beguiling freshness,1
2,"'s dull , spiritless , silly and monotonous",0
3,beginning with the minor omission of a screenplay,0
4,is one of world cinema 's most wondrously gift...,1


In [4]:
texts = df['text'].tolist()
labels = df['label'].tolist()

print(f'Текстов: {len(texts)}, меток: {len(labels)}')
for t, l in zip(texts[:3], labels[:3]):
    print(f'  [{l}] {t[:70]}')

Текстов: 3000, меток: 3000
  [1] a perfect family film
  [1] brings a beguiling freshness
  [0] 's dull , spiritless , silly and monotonous


### 3.2 Извлечение эмбеддингов

Самый долгий шаг — 3000 текстов через DistilBERT на CPU (примерно минута-две).

In [5]:
X = get_cls_embeddings(texts)
print(f'Embeddings shape: {X.shape}')

Embeddings shape: (3000, 768)


### 3.3 Train/test split и обучение

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, stratify=labels, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

clf = LogisticRegression(max_iter=1000, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('Обучено.')

Train: (2400, 768), Test: (600, 768)
Обучено.


C:\Users\Vsevolod\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


### 3.4 Метрики

In [7]:
print(classification_report(y_test, y_pred, target_names=['negative', 'positive']))

f1 = f1_score(y_test, y_pred, average='macro')
print(f'macro F1: {f1:.4f}')

              precision    recall  f1-score   support

    negative       0.85      0.84      0.85       265
    positive       0.88      0.88      0.88       335

    accuracy                           0.86       600
   macro avg       0.86      0.86      0.86       600
weighted avg       0.86      0.86      0.86       600

macro F1: 0.8630


### 3.5 Сохранение результатов

In [8]:
from baseline_utils import train_baseline, save_results

# train_baseline повторяет шаги 3.3-3.4 одной функцией (тот же random_state=42)
result = train_baseline(X, labels)
save_results('baseline_results.txt', result)

print(f'macro F1: {result.f1_macro:.4f}')
print('\n--- baseline_results.txt ---')
print(open('baseline_results.txt', encoding='utf-8').read())

macro F1: 0.8630

--- baseline_results.txt ---
Baseline без обучения трансформера (День 4)
Модель (заморожена):  distilbert-base-uncased
Признаки:             CLS-эмбеддинги, last_hidden_state[:, 0, :]
Классификатор:        LogisticRegression(max_iter=1000)
Датасет:              SST-2
Train / test:         2400 / 600

macro F1: 0.8630

Classification report:
              precision    recall  f1-score   support

    negative       0.85      0.84      0.85       265
    positive       0.88      0.88      0.88       335

    accuracy                           0.86       600
   macro avg       0.86      0.86      0.86       600
weighted avg       0.86      0.86      0.86       600



C:\Users\Vsevolod\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


## Итог

Baseline на замороженных CLS-эмбеддингах даёт **macro F1 ≈ 0.86** на SST-2.

Что это значит:
- Трансформер **не обучался** — веса остались предобученными, градиенты не считались (`torch.no_grad()`). Обучилась только логистическая регрессия: 768 весов + свободный член.
- Тем не менее CLS-эмбеддинги уже несут достаточно информации о тональности, чтобы линейный классификатор разделил классы с точностью 86%.
- Это и есть **точка отсчёта** для fine-tuning. Дообучение всей модели на SST-2 обычно даёт ~0.90+, так что baseline стоит превзойти заметно, иначе смысла в дообучении нет.

Полезная деталь для Дня 2: там косинусное сходство «сырых» эмбеддингов почти не различало тональность (0.995 против 0.984). Здесь видно, почему — информация в эмбеддингах **есть**, просто она не лежит вдоль направления косинусной близости. Логистическая регрессия находит нужное направление в 768-мерном пространстве, а косинус усредняет по всем измерениям сразу.